In [1]:
import numpy as np
import polars as pl
from pathlib import Path

from src.models.data import fp_from_smiles
from src.models.pipeline import results_table, train_and_score, tune_gnn
from src.models import config as model_cfg
from src._config import PROCESSED_DATA

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

/home/computer/Repositories/ml_chembl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

In [2]:
df = pl.read_parquet(PROCESSED_DATA / "ChEMBL_processed.parquet")

In [3]:
print(df.columns)

['activity_id', 'molregno', 'canonical_smiles', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'aromatic_rings', 'qed_weighted', 'standard_value', 'standard_units', 'standard_type', 'standard_relation', 'pchembl_value', 'target_chembl_id', 'target_name', 'confidence_score', 'pIC50']


In [4]:
# Modele MLPBaseline i GNNRegressor przeniesione do src/models/models.py

In [5]:
# seed_everything, get_device, compute_num_workers przeniesione do src/models/training.py

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti


In [6]:
# Funkcje treningowe (train_one_epoch, evaluate_loss, evaluate_r2) przeniesione do src/models/training.py

In [7]:
df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(fp_from_smiles, return_dtype=pl.Object).alias("fp")
)
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba próbek po pełnym oczyszczeniu: {len(df_clean)}")
print(f"Odrzucono {len(df) - len(df_clean)} błędnych cząsteczek.")

Liczba próbek po pełnym oczyszczeniu: 15723
Odrzucono 0 błędnych cząsteczek.


In [8]:
# Funkcje splitow, grafow i loaderow przeniesione do src/models/data.py

In [9]:
# Funkcje splitow, grafow i loaderow przeniesione do src/models/data.py

In [10]:
# Konfiguracja domyslnych parametrow – przeniesiona do src/models/config.py
from src.models.config import (
    EPOCHS_DEFAULT, LR_DEFAULT, BATCH_SIZE_DEFAULT, SEED_DEFAULT,
    EARLY_STOPPING_PATIENCE_DEFAULT, MIN_DELTA_DEFAULT, WEIGHT_DECAY_DEFAULT, POOLING_DEFAULT,
)
print(
    f'Domyslne parametry: epochs={EPOCHS_DEFAULT}, lr={LR_DEFAULT}, batch={BATCH_SIZE_DEFAULT}, '
    f'seed={SEED_DEFAULT}, patience={EARLY_STOPPING_PATIENCE_DEFAULT}, pooling={POOLING_DEFAULT}'
)


Domyslne parametry: epochs=100, lr=0.0003, batch=64, seed=42, patience=12, pooling=mean


In [11]:
fp_has_nan = any(np.isnan(fp).any() for fp in df_clean["fp"])
print(f"NaN w fingerprintach: {fp_has_nan}")

NaN w fingerprintach: False


In [12]:
# Funkcje train_and_score i tune_gnn przeniesione do src/models/pipeline.py
# results_table, train_and_score, tune_gnn zaimportowane w komorce 0

In [13]:
# MLP - random split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="random", df_fp=df_clean, log_mlflow=True, evaluate_test=False)

Loaded cached model: default_37352a75cf5257ec.pt | val R2=0.737


{'model': 'MLP',
 'split': 'random',
 'seed': 42,
 'epochs': 100,
 'epochs_trained': 73,
 'best_epoch': 61,
 'lr': 0.0003,
 'batch_size': 64,
 'weight_decay': 1e-05,
 'pooling': 'mean',
 'gnn_hidden_dim': 128,
 'gnn_num_layers': 4,
 'gnn_dropout': 0.15,
 'avg_train_loss': 0.42569581716972965,
 'avg_val_loss': 0.48538488060644236,
 'best_val_loss': 0.45433297991752625,
 'r2_val': 0.736980676651001,
 'r2_test': None,
 'device': 'cuda',
 'amp_enabled': True,
 'from_cache': True,
 'cache_path': 'processed_data/model_cache/default_37352a75cf5257ec.pt'}

In [14]:
# MLP - scaffold split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="scaffold", df_fp=df_clean, log_mlflow=True, evaluate_test=False)

Loaded cached model: default_1e675e4c71a847e4.pt | val R2=0.447


{'model': 'MLP',
 'split': 'scaffold',
 'seed': 42,
 'epochs': 100,
 'epochs_trained': 17,
 'best_epoch': 5,
 'lr': 0.0003,
 'batch_size': 64,
 'weight_decay': 1e-05,
 'pooling': 'mean',
 'gnn_hidden_dim': 128,
 'gnn_num_layers': 4,
 'gnn_dropout': 0.15,
 'avg_train_loss': 0.7984636863704776,
 'avg_val_loss': 0.7767915284633637,
 'best_val_loss': 0.6543190622329712,
 'r2_val': 0.4469124674797058,
 'r2_test': None,
 'device': 'cuda',
 'amp_enabled': True,
 'from_cache': True,
 'cache_path': 'processed_data/model_cache/default_1e675e4c71a847e4.pt'}

In [15]:
# GNN - uruchamiamy tylko 1 najlepsza konfiguracje na split
# Dla oceny 4.0 logujemy kazdy finalny run do MLflow.
BEST_GNN_CONFIGS = {
    'scaffold': {
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'pooling': 'mean',
        'gnn_hidden_dim': 192,
        'gnn_num_layers': 4,
        'gnn_dropout': 0.10,
    },
    'random': {
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'pooling': 'mean',
        'gnn_hidden_dim': 192,
        'gnn_num_layers': 4,
        'gnn_dropout': 0.10,
    },
}

gnn_selected_results = {}
for split_type, cfg in BEST_GNN_CONFIGS.items():
    gnn_selected_results[split_type] = train_and_score(
        model_type='GNN',
        split_type=split_type,
        df_fp=df_clean,
        epochs=100,
        batch_size=64,
        seed=42,
        log_mlflow=True,
        evaluate_test=False,
        early_stopping_patience=12,
        min_delta=1e-4,
        prefer_cuda=True,
        **cfg,
    )

gnn_selected_results


Loaded cached model: default_bc35dbdc16a2e430.pt | val R2=0.461
Loaded cached model: default_5d174cf5636aa4ea.pt | val R2=0.649


{'scaffold': {'model': 'GNN',
  'split': 'scaffold',
  'seed': 42,
  'epochs': 100,
  'epochs_trained': 62,
  'best_epoch': 50,
  'lr': 0.0003,
  'batch_size': 64,
  'weight_decay': 1e-05,
  'pooling': 'mean',
  'gnn_hidden_dim': 192,
  'gnn_num_layers': 4,
  'gnn_dropout': 0.1,
  'avg_train_loss': 0.8979609732744802,
  'avg_val_loss': 1.0461740172678424,
  'best_val_loss': 0.7915135335922241,
  'r2_val': 0.4610564112663269,
  'r2_test': None,
  'device': 'cuda',
  'amp_enabled': True,
  'from_cache': True,
  'cache_path': 'processed_data/model_cache/default_bc35dbdc16a2e430.pt'},
 'random': {'model': 'GNN',
  'split': 'random',
  'seed': 42,
  'epochs': 100,
  'epochs_trained': 81,
  'best_epoch': 69,
  'lr': 0.0003,
  'batch_size': 64,
  'weight_decay': 1e-05,
  'pooling': 'mean',
  'gnn_hidden_dim': 192,
  'gnn_num_layers': 4,
  'gnn_dropout': 0.1,
  'avg_train_loss': 0.8396239752977005,
  'avg_val_loss': 0.8469091184492463,
  'best_val_loss': 0.6075526428222656,
  'r2_val': 0.64925

In [16]:
# Podsumowanie wybranych konfiguracji GNN
import pandas as pd

if 'gnn_selected_results' in globals() and gnn_selected_results:
    cfg_df = pd.DataFrame([
        {'split': split, **cfg}
        for split, cfg in BEST_GNN_CONFIGS.items()
    ])
    score_df = pd.DataFrame([
        {'split': split, 'r2_val': res['r2_val'], 'best_val_loss': res['best_val_loss']}
        for split, res in gnn_selected_results.items()
    ])
    display(cfg_df.merge(score_df, on='split', how='left'))
else:
    print('Uruchom najpierw komorke z BEST_GNN_CONFIGS.')


,split,lr,weight_decay,pooling,gnn_hidden_dim,gnn_num_layers,gnn_dropout,r2_val,best_val_loss
0,scaffold,0.0003,0.00001,mean,192,4,0.1,0.461056,0.791514
1,random,0.0003,0.00001,mean,192,4,0.1,0.649254,0.607553


In [17]:
# Zestawienie wynikow w tabeli
import pandas as pd

if not results_table:
    print('Brak zebranych wynikow. Uruchom najpierw komorki treningowe.')
else:
    df_results = pd.DataFrame(results_table)
    preferred_cols = [
        'model',
        'split',
        'seed',
        'epochs',
        'epochs_trained',
        'best_epoch',
        'lr',
        'batch_size',
        'weight_decay',
        'pooling',
        'gnn_hidden_dim',
        'gnn_num_layers',
        'gnn_dropout',
        'avg_train_loss',
        'avg_val_loss',
        'best_val_loss',
        'r2_val',
        'r2_test',
        'from_cache',
        'cache_path',
    ]
    cols_to_show = [c for c in preferred_cols if c in df_results.columns]
    display(df_results[cols_to_show].sort_values(['r2_val'], ascending=False).reset_index(drop=True))


,model,split,seed,epochs,epochs_trained,best_epoch,lr,batch_size,weight_decay,pooling,gnn_hidden_dim,gnn_num_layers,gnn_dropout,avg_train_loss,avg_val_loss,best_val_loss,r2_val,r2_test,from_cache,cache_path
0,MLP,random,42,100,73,61,0.0003,64,0.00001,mean,128,4,0.15,0.425696,0.485385,0.454333,0.736981,None,True,processed_data/model_cache/default_37352a75cf5...
1,GNN,random,42,100,81,69,0.0003,64,0.00001,mean,192,4,0.10,0.839624,0.846909,0.607553,0.649254,None,True,processed_data/model_cache/default_5d174cf5636...
2,GNN,scaffold,42,100,62,50,0.0003,64,0.00001,mean,192,4,0.10,0.897961,1.046174,0.791514,0.461056,None,True,processed_data/model_cache/default_bc35dbdc16a...
3,MLP,scaffold,42,100,17,5,0.0003,64,0.00001,mean,128,4,0.15,0.798464,0.776792,0.654319,0.446912,None,True,processed_data/model_cache/default_1e675e4c71a...


In [18]:
# Finalny test uruchom raz dla najlepszego wariantu wybranego po walidacji.
# Przyklad: pobierz top konfiguracje i odpal dluzszy trening na 1 seed z testem.
# best_cfg = gnn_scaffold_tuning.iloc[0].to_dict()
# final_result = train_and_score(
#     model_type='GNN',
#     split_type='scaffold',
#     epochs=100,
#     lr=best_cfg['lr'],
#     batch_size=64,
#     seed=42,
#     weight_decay=best_cfg['weight_decay'],
#     pooling=best_cfg['pooling'],
#     gnn_hidden_dim=int(best_cfg['gnn_hidden_dim']),
#     gnn_num_layers=int(best_cfg['gnn_num_layers']),
#     gnn_dropout=best_cfg['gnn_dropout'],
#     log_mlflow=True,
#     evaluate_test=True,
#     replace_existing=False,
#     prefer_cuda=True,
#     deterministic=False,
#     use_amp=True,
# )


# Wnioski z porownania modeli

- Ranking wariantow wykonuj na podstawie **R2 walidacyjnego**, a zbior testowy uruchamiaj tylko raz dla finalisty.
- Dla GNN raportuj srednia i odchylenie standardowe po wielu seedach (minimum 3).
- Scaffold split traktuj jako glowny test uogolniania chemicznego na nowe rusztowania.
- Najpierw wybierz najlepsza konfiguracje po `gnn_scaffold_tuning`, dopiero potem uruchom finalny test.
